# 📈 Nifty500 Stock Direction Prediction

**Goal:** Predict whether a stock's closing price will go **UP or DOWN** the next trading day.

**Dataset:** Nifty500 5-Year Historical OHLCV data (2021–2026) — 550K+ rows across 500 Indian stocks.

**What you'll learn:**
- Exploratory Data Analysis on stock market data
- Feature Engineering with technical indicators (RSI, MA, Volatility)
- How to avoid **data leakage** in time-series ML
- Binary classification with Random Forest & XGBoost
- Model evaluation using Accuracy, F1-Score & Confusion Matrix

---

### 📋 Notebook Structure
1. Import Libraries  
2. Load & Explore Data  
3. Feature Engineering (Technical Indicators)  
4. Time-Based Train-Test Split  
5. Model Training  
6. Model Evaluation  
7. Feature Importance  
8. Conclusion  

## 1️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ All libraries imported!')

## 2️⃣ Load & Explore Data (EDA)

> 💡 **Tip:** Always explore your data before writing any model code!

In [ ]:
df = pd.read_csv('nifty500_5yr_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f'📊 Shape       : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'📅 Date Range  : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'🏢 Unique Stocks: {df["Ticker"].nunique()}')
df.head()

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())

# Drop 'Notes' — mostly empty, adds no value
df.drop(columns=['Notes'], inplace=True)
print('\n✅ Dropped Notes column.')

In [ ]:
df.describe().round(2)

In [ ]:
# Plot closing price of a few well-known stocks
sample_tickers = ['RELIANCE', 'TCS', 'INFY', 'HDFCBANK', 'ITC']
available = [t for t in sample_tickers if t in df['Ticker'].values]
sample_df = df[df['Ticker'].isin(available)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ticker in available:
    temp = sample_df[sample_df['Ticker'] == ticker].sort_values('Date')
    axes[0].plot(temp['Date'], temp['Close'], label=ticker)
axes[0].set_title('Closing Price Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Price (₹)')
axes[0].legend()

axes[1].hist(np.log1p(df['Volume']), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Log Volume Distribution (All 500 Stocks)')
axes[1].set_xlabel('log(Volume + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3️⃣ Feature Engineering

Raw OHLCV columns are not enough. We create **technical indicators** that traders and quants use.

| Feature | What it captures |
|---|---|
| `return_1d / 3d / 5d / 10d` | Price momentum |
| `ma_ratio_5_20`, `ma_ratio_20_50` | Short vs long-term trend |
| `price_vs_ma20`, `price_vs_ma50` | Is price above/below average? |
| `vol_5`, `vol_20` | How volatile is the stock? |
| `volume_ratio`, `vol_trend` | Unusual trading activity |
| `high_low_pct`, `open_close_pct` | Intraday strength |
| `upper_shadow`, `lower_shadow` | Candle pattern signals |
| `rsi_14` | Overbought / oversold? |
| `lag1/2/3_return` | Yesterday's momentum |
| `day_of_week`, `month` | Seasonality patterns |

**⚠️ Key rule:** Features are computed **per ticker** using `groupby` — we never mix data between different stocks.

**Target:** `1` if tomorrow's Close > today's Close, else `0`.

In [ ]:
def add_features(group):
    """Compute all features for one stock independently."""
    g = group.sort_values('Date').copy()

    # Price returns (momentum)
    g['return_1d']  = g['Close'].pct_change(1)
    g['return_3d']  = g['Close'].pct_change(3)
    g['return_5d']  = g['Close'].pct_change(5)
    g['return_10d'] = g['Close'].pct_change(10)

    # Moving averages & ratios
    g['ma_5']  = g['Close'].rolling(5).mean()
    g['ma_20'] = g['Close'].rolling(20).mean()
    g['ma_50'] = g['Close'].rolling(50).mean()
    g['ma_ratio_5_20']  = g['ma_5']  / g['ma_20']
    g['ma_ratio_20_50'] = g['ma_20'] / g['ma_50']
    g['price_vs_ma20']  = g['Close'] / g['ma_20']
    g['price_vs_ma50']  = g['Close'] / g['ma_50']

    # Volatility
    g['vol_5']  = g['return_1d'].rolling(5).std()
    g['vol_20'] = g['return_1d'].rolling(20).std()

    # Volume features
    g['vol_ma_5']     = g['Volume'].rolling(5).mean()
    g['vol_ma_20']    = g['Volume'].rolling(20).mean()
    g['volume_ratio'] = g['Volume'] / g['vol_ma_5']
    g['vol_trend']    = g['vol_ma_5'] / g['vol_ma_20']

    # Candle body features
    g['high_low_pct']   = (g['High'] - g['Low']) / g['Close']
    g['open_close_pct'] = (g['Close'] - g['Open']) / g['Open']
    g['upper_shadow']   = (g['High'] - g[['Open','Close']].max(axis=1)) / g['Close']
    g['lower_shadow']   = (g[['Open','Close']].min(axis=1) - g['Low']) / g['Close']

    # RSI (14-day)
    delta = g['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    g['rsi_14'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # Lagged returns (yesterday, day before, etc.)
    g['lag1_return'] = g['return_1d'].shift(1)
    g['lag2_return'] = g['return_1d'].shift(2)
    g['lag3_return'] = g['return_1d'].shift(3)

    # Seasonality
    g['day_of_week'] = g['Date'].dt.dayofweek
    g['month']       = g['Date'].dt.month

    # TARGET: will tomorrow be higher?
    # shift(-1) = next row's value, so we're predicting the future
    g['target'] = (g['Close'].shift(-1) > g['Close']).astype(int)

    return g

print('⚙️  Engineering features for 500 stocks... (~30 sec)')
df = df.groupby('Ticker', group_keys=False).apply(add_features)
df.dropna(inplace=True)  # Remove rows with NaN from rolling windows
print(f'✅ Done! Shape: {df.shape}')

In [ ]:
# Target class balance
tc = df['target'].value_counts()
print(f'UP   (1): {tc[1]:,} ({tc[1]/len(df)*100:.1f}%)')
print(f'DOWN (0): {tc[0]:,} ({tc[0]/len(df)*100:.1f}%)')

plt.figure(figsize=(6, 4))
df['target'].value_counts().plot(
    kind='bar', color=['#e74c3c','#2ecc71'], edgecolor='black'
)
plt.xticks([0,1], ['DOWN (0)','UP (1)'], rotation=0)
plt.title('Target: UP vs DOWN Days')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4️⃣ Time-Based Train-Test Split

> ⚠️ **Do NOT use random splitting on time-series data!**
> 
> If you train on 2024 data and test on 2023 data — that's **cheating**. The model learns the future and looks artificially great.
> 
> **Correct approach:** Train on older data → Test on newer data.

In [ ]:
FEATURES = [
    'return_1d', 'return_3d', 'return_5d', 'return_10d',
    'ma_ratio_5_20', 'ma_ratio_20_50', 'price_vs_ma20', 'price_vs_ma50',
    'vol_5', 'vol_20', 'volume_ratio', 'vol_trend',
    'high_low_pct', 'open_close_pct', 'upper_shadow', 'lower_shadow',
    'rsi_14', 'lag1_return', 'lag2_return', 'lag3_return',
    'day_of_week', 'month'
]

# Sort chronologically — critical for time-series
df = df.sort_values('Date')

# 80% train, 20% test — by date
split_date = df['Date'].quantile(0.80)
train_df   = df[df['Date'] <= split_date]
test_df    = df[df['Date']  > split_date]

X_train, y_train = train_df[FEATURES], train_df['target']
X_test,  y_test  = test_df[FEATURES],  test_df['target']

print(f'🗓️  Train: {train_df["Date"].min().date()} → {train_df["Date"].max().date()} | {len(X_train):,} rows')
print(f'🗓️  Test : {test_df["Date"].min().date()} → {test_df["Date"].max().date()} | {len(X_test):,} rows')

## 5️⃣ Model Training

We train two powerful ensemble models:

- **Random Forest** — builds many decision trees independently and votes
- **XGBoost** — builds trees sequentially, each one fixing the errors of the previous

In [ ]:
# Random Forest
print('🌲 Training Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_preds)
print(f'✅ RF Accuracy  : {rf_acc:.4f} ({rf_acc*100:.2f}%)')

In [ ]:
# XGBoost
print('⚡ Training XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=50,
    gamma=0.1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_acc   = accuracy_score(y_test, xgb_preds)
print(f'✅ XGB Accuracy : {xgb_acc:.4f} ({xgb_acc*100:.2f}%)')

## 6️⃣ Model Evaluation

> 📌 In stock market prediction, even **51-55% accuracy is meaningful** — it's statistically better than random guessing (50%) across 100K+ predictions.
> Professional quant models often target 52-55% and are highly profitable with proper position sizing.

In [ ]:
# Compare accuracy
models = ['Random Forest', 'XGBoost']
scores = [rf_acc, xgb_acc]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, scores, color=['#3498db','#e67e22'], edgecolor='black', width=0.4)
plt.axhline(0.50, color='red', linestyle='--', label='Random Baseline (50%)')
plt.ylim(0.48, 0.58)
plt.title('Model Accuracy vs Random Baseline')
plt.ylabel('Accuracy')
plt.legend()
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{score:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Use the better model
if xgb_acc >= rf_acc:
    best_preds, best_name = xgb_preds, 'XGBoost'
else:
    best_preds, best_name = rf_preds, 'Random Forest'

print(f'🏆 Best Model: {best_name}\n')
print(classification_report(y_test, best_preds, target_names=['DOWN','UP']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['DOWN','UP'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## 7️⃣ Feature Importance

Which features helped the model most? This tells us what the model "learned" to look at.

In [ ]:
importance_df = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 7))
colors = ['#e74c3c' if v > importance_df['Importance'].median() else '#3498db'
          for v in importance_df['Importance']]
plt.barh(importance_df['Feature'], importance_df['Importance'],
         color=colors, edgecolor='white')
plt.title('XGBoost Feature Importance (Red = Above Median)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('🔝 Top 5 Features:')
print(importance_df.tail(5)[['Feature','Importance']].to_string(index=False))

## 8️⃣ Conclusion

### 📝 Summary

We built a **next-day stock direction classifier** on the Nifty500 5-year OHLCV dataset.

| Step | What We Did |
|---|---|
| **EDA** | Explored 550K+ rows, 500 tickers over 5 years |
| **Feature Engineering** | Created 22 technical indicators per stock |
| **Leakage-Free Split** | Date-based 80/20 split — no shuffling! |
| **Models** | Trained Random Forest and XGBoost |
| **Evaluation** | Compared accuracy, F1, confusion matrix |

### 📊 Results

Both models achieve **~51-53% accuracy** — consistently above the 50% random baseline.

> 🧠 **Why isn't accuracy higher?**  
> Stock markets are influenced by earnings surprises, global news, RBI policy, FII flows — none of which are in OHLCV data alone. A 51-55% edge from pure price data is actually impressive and aligns with academic research on market predictability.

### 💡 Key Takeaways
- **Data leakage** is the #1 mistake in time-series ML — never shuffle before splitting
- **Feature engineering** matters more than model choice in stock prediction
- **RSI, price vs MA, and lagged returns** carry the most signal
- Even small accuracy edges (~51%) can be profitable at scale with proper risk management

### 🚀 What to Try Next
1. Add **MACD, Bollinger Bands, ATR** as features
2. Build **per-sector models** (IT, Banking, Pharma separately)
3. Try **LSTM / Transformer** models for sequence learning
4. Incorporate **Nifty50 index returns** as a market feature
5. Add **fundamental data** (P/E ratio, earnings) for better signals

---
*If you found this notebook useful, please upvote ⬆️ — it helps others discover it!*